<h3>Importing the Libraries

In [4]:
import os
import cv2
from pathlib import Path
import argparse
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

import torchvision
import torchvision.transforms as transforms

from efficientnet_pytorch import EfficientNet

In [2]:
for file_name in os.listdir():
    name, extension = os.path.splitext(file_name)
    if extension == '.png':
        original_image = cv2.imread(file_name)
        original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY)
        resized_image = cv2.resize(original_image, (100, 100))
        cv2.imwrite(file_name, resized_image)

In [15]:
def class_initiator(folder):
    file_array = []
    for file_name in folder.iterdir():
        # name, extension = os.path.splitext(file_name)
        if file_name.suffix == '.png':
            file_array.append(file_name.name)
    return file_array

In [4]:
file_array = class_initiator()
class_dictionary = {
            'file_name': file_array
          }
df = pd.DataFrame(class_dictionary)
df.to_csv('dataset.csv')

In [24]:
class CustomDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform = None):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, index):
        img_path = os.path.join(str(self.root_dir), str(self.annotations.iloc[index, 1]))
        image = plt.imread(img_path)
        
        if self.transform:
            image = self.transform(image)
        return (image)

In [52]:
ROOT = Path().cwd().parent
DATA_DIR = ROOT / "data" / "stimuli" / "shape_dataset"

dataset = CustomDataset(csv_file = DATA_DIR / "dataset.csv", root_dir = DATA_DIR / "resized", transform = transforms.ToTensor())
batch_size = 500
test_loader = DataLoader(dataset = dataset, batch_size = batch_size, shuffle = True)

In [53]:
file_array = class_initiator(DATA_DIR / "resized")

In [54]:
len(file_array)

90000

<h3>Neural Network Class

In [29]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.conv3 = nn.Conv2d(64, 128, 5, 1)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(10368, 4096)
        self.fc2 = nn.Linear(4096, 1024)
        self.fc3 = nn.Linear(1024, 128)
        self.fc4 = nn.Linear(128, 9)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)

        return x

In [30]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = (1e-3))

In [31]:
model.load_state_dict(torch.load(ROOT / "data" / "models" / "cnn_model.pt"))

<All keys matched successfully>

In [77]:
preds = []
for values in test_loader:
    values = values.to(device)
    output = model(values)
    pred = output.max(1)
    # pred = output.argmax(dim=1, keepdim=True)
    preds.append(pred)

ValueError: too many values to unpack (expected 2)

In [78]:
values.shape

torch.Size([500, 1, 100, 100])

In [79]:
output.shape

torch.Size([500, 9])

In [80]:
pred.shape

torch.Size([500, 1])

In [81]:
pred

tensor([[3],
        [4],
        [7],
        [5],
        [5],
        [7],
        [3],
        [1],
        [4],
        [8],
        [8],
        [4],
        [0],
        [4],
        [4],
        [1],
        [0],
        [5],
        [7],
        [1],
        [5],
        [2],
        [7],
        [4],
        [2],
        [4],
        [3],
        [6],
        [8],
        [5],
        [1],
        [6],
        [8],
        [7],
        [1],
        [0],
        [1],
        [0],
        [8],
        [5],
        [4],
        [3],
        [1],
        [8],
        [8],
        [1],
        [8],
        [2],
        [8],
        [2],
        [4],
        [6],
        [4],
        [7],
        [2],
        [8],
        [0],
        [2],
        [8],
        [8],
        [6],
        [2],
        [1],
        [3],
        [0],
        [0],
        [4],
        [0],
        [0],
        [4],
        [6],
        [3],
        [7],
        [0],
        [2],
        [7],
        [2],

In [35]:
count = 0
correct = 0
for output_value in preds:
    print("File Name: ", file_array[count])
    if output_value.item() == 0:
        pred_name = "Circle"
        print("Circle")
    if output_value.item() == 1:
        print("Square")
        pred_name = "Square"
    if output_value.item() == 2:
        print("Octagon")
        pred_name = "Octagon"
    if output_value.item() == 3:
        print("Heptagon")
        pred_name = "Heptagon"
    if output_value.item() == 4:
        print("Nonagon")
        pred_name = "Nonagon"

    if output_value.item() == 5:
        print("Star")
        pred_name = "Star"

    if output_value.item() == 6:
        print("Hexagon")
        pred_name = "Hexagon"
    if output_value.item() == 7:
        print("Pentagon")
        pred_name = "Pentagon"

    if output_value.item() == 8:
        print("Triangle")
        pred_name = "Triangle"

    if pred_name in file_array[count]:
        correct+=1

    count+=1

print("Accuracy: ", correct/len(file_array))


NameError: name 'preds' is not defined

In [24]:
model

Net(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1))
  (dropout1): Dropout(p=0.25, inplace=False)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=10368, out_features=4096, bias=True)
  (fc2): Linear(in_features=4096, out_features=1024, bias=True)
  (fc3): Linear(in_features=1024, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=9, bias=True)
)